# 22 — XGBoost sobre el índice compartido (auto-generado)

Generado por `build_notebook_22_xgb_contiguous.py`. **No editar a mano.**

Reajusta el baseline B5_XGB sobre **exactamente la misma población** que consume
el LSTM reentrenado (notebooks 21). La garantía no es un archivo compartido: cada
kernel recomputa el índice canónico desde el mismo parquet hash-pineado y
verifica que su SHA-256 coincida con `sample_index_manifest.csv`. Mismo código
más mismos bytes da el mismo índice.

Con `N_LAGS == T_IN == 12`, el XGBoost ve **las mismas doce observaciones** que
la red. Nivelar deja de ser un argumento y pasa a ser una propiedad verificable.

**Qué cambia respecto de NB10/NB16.** Los lags ya no se construyen con
`shift()` posicional sobre el slot —que nunca verificó contigüidad temporal— sino
leyéndolos de la ventana contigua. Y se retira la bandera de día atípico: es un
agregado del día completo y por lo tanto no se conoce al momento de predecir.

**La búsqueda de 24 configuraciones se conserva sin cambios** (`SEARCH_SEED`,
`SEARCH_SPACE`), seleccionada estrictamente sobre validación.

In [ ]:

import hashlib, time

import numpy as np
import polars as pl
import xgboost as xgb
from pathlib import Path

INPUT_HASHES = {
    "headways_E2.parquet": "82a34eaffc79cd82346d4595a2e72f5d3ffb751ed37fa0fc0cde3a8f8fb345d4",
    "headways_E59.parquet": "0b5f5593caaa94e4e6af7da672bc2cad7b49b69b7cbd0a22092f15700a89a448",
    "headways_E4.parquet": "1dde7f38eea9bc7d9941c17cbc3d326cb864e70be815a1a7e3d0ae2691f19273"
}

INDEX_DIGESTS = {
    "E2|train|1": "f4c6e5df773197cfd7fb85512346018a9857f4cde87af7e71aa52cb4f232710e",
    "E2|train|3": "2c996b641705ac08bdc3b4fe73c5079161da34b93da327c138de4bdfc459846a",
    "E2|train|5": "41616abe14d433dde5d7da797f5ef3bfefc2f486e225e7889002f9e4fbdbbe1c",
    "E2|train|10": "db067c45842f47d1491ecfc37a581cfbbcf5b01678e2a0bc83a3fea3d74d86e5",
    "E2|val|1": "f5be1c8aa832abc0d4dd412ed7605b14fc5e8028b6a70755edb9154613008222",
    "E2|val|3": "ea8fb75fb2f0af88213dc6a936bb6c7e11a1615740e679e43e4432554e53e869",
    "E2|val|5": "8a92631d69e8094485cd02c1e894f9fc29ae0e556dfe284fe7e5fe2f5e71f591",
    "E2|val|10": "53710d90a1967f455ed1e34ff0cff27935c19dbcfc2dd58c0919752f37fbe4b5",
    "E2|test|1": "ab28518fb3165c769b89856786faf324920e120256aec1576a4d63c96ba3701f",
    "E2|test|3": "585114ada9eec08d3a7b492b99d3bdb45bfee0dd3dba84a45e48051c64b4b04c",
    "E2|test|5": "78e9257302c52ffbd4554988af073791c19f4e04b4153b7e083e05196284551b",
    "E2|test|10": "918554479e9a42caac530706e0a9f0d6841240e02445d8406d5156a6df05eec2",
    "E59|train|1": "1eaec9bc2d42cf9e30f92bcc703a899f08f406dda24ee65cedcd52e6291b85c7",
    "E59|train|3": "f4746e34d4ff1dcc763c73cdba481183c71bbbe4d801e9cfb6ac4d36a2a047f4",
    "E59|train|5": "80ee5736b248f51a8eb11bffba27b05b56aae86c364ee0b4d34ed687587a3a7b",
    "E59|train|10": "ecdafc0780563875bd8464c2b14829dccf6caa854f5bfd73e92ff9f9b94b7324",
    "E59|val|1": "43c292b4a705c75179f2fbc1df4ccbf139c7e67484c08f779cc79ec2a289eac7",
    "E59|val|3": "e4e01f714831a53de700a55ea6ad326d4e588b42dd38f83989b42f3f76e25153",
    "E59|val|5": "9946a06324a9afdb96de25e11c946051e9ab8d40d9a32f250e8cbf4d5782f4b6",
    "E59|val|10": "6ea277b0f77b0bdbda2577c1090b8fb6103569263dda47d774cf2c4f471019a3",
    "E59|test|1": "008f5ff38c23699a752d65db5bc6486364a3b34950b915443642a8243db0e460",
    "E59|test|3": "342a6a5d0a07fd21442e3e3aa30bedd1c11e1d1bb5b7d5ebcf7490f4fca54858",
    "E59|test|5": "bb2b632f5c61851a42baa2a46ba34c25df88b3f9edba520852eca8568d1469fe",
    "E59|test|10": "59809befeb6c188095b58057b41d892ca43413059d25d3da51e54e9da32da3ec",
    "E4|train|1": "13aa3843ebe44a6dd1653365ccbbbfb5820911fd54ae2cc2923a4167d0bae9b0",
    "E4|train|3": "3c51fa201c291f88c23c659151dac2549ca7a7658477384ceb2a8458c6d7dd4e",
    "E4|train|5": "b3bd4bdfca8b53e5db61e3da86a78d5891ccaf0858a7375035138c94a8cf3808",
    "E4|train|10": "9bbcafbe34838647457b3fa5595f07205b03d51d1e9d429262e190b637fcd407",
    "E4|val|1": "898d18fad978ed48d05944038c6627b5e3ecd9d3dd078ce18931f995d470531b",
    "E4|val|3": "eb35aa062e35e13db3098d4ec0bf0bee6dded9b59ac69647edea071bc8e52e72",
    "E4|val|5": "221d160532ac68f38bcde8986f7625073dc171fe7d24f26d2e42410e1e600722",
    "E4|val|10": "cf449d4a61b87d8bb1aec3839d1cf2ab3ac094540a882964352559788a804f15",
    "E4|test|1": "906d111281e491ed7f9a1771856fd125b25bd1d8a7f944ba036d2b85aefaf27b",
    "E4|test|3": "5244ebd04d6175f204f4b8cfc8cab234baf34a8043a5588bda099d574d126ef5",
    "E4|test|5": "2a2da36998abd23cb49c96d3a77063f8a618a8adde02554bf4b2eb10190f70ca",
    "E4|test|10": "d0ccd8ea318518b6da8eb7c7a0fce6539c599c7b45d4d4cbcb0681eaf51fc4c8"
}

CORRIDORS = [["E2", 2], ["E59", 59], ["E4", 4]]
HORIZONS = [1, 3, 5, 10]

def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def _resolve_input(name: str) -> Path:
    roots = [Path("/kaggle/input"), Path(".")]
    candidates = [p for root in roots if root.exists() for p in sorted(root.rglob(name))]
    if not candidates:
        raise FileNotFoundError(f"Required input not found anywhere: {name}")
    for path in candidates:
        if _sha256_file(path) == INPUT_HASHES[name]:
            return path
    raise ValueError(
        f"No copy of {name} matches its frozen SHA-256 — "
        f"candidates: {[str(p) for p in candidates]}"
    )

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUTPUT_DIR.mkdir(exist_ok=True)
RESULTS_OUT = OUTPUT_DIR / "xgb_contig_results.csv"
RESID_OUT = OUTPUT_DIR / "xgb_contig_residuals.csv"
SEARCH_OUT = OUTPUT_DIR / "xgb_contig_search_config.csv"
print(f"Output dir: {OUTPUT_DIR}")

## Module: evaluation/splits

`split_temporal` + `winsorize_train_p99` (umbral solo de train, aplicado a todos los splits).

In [ ]:
"""Temporal split and winsorization helpers for headway evaluation — Fase 3.

Public API:
    split_temporal(df: pl.DataFrame, fold: Fold = MAIN_FOLD) -> pl.DataFrame
    winsorize_train_p99(df: pl.DataFrame) -> tuple[pl.DataFrame, float]
    Fold, MAIN_FOLD, ROLLING_FOLDS, fold_by_name

Constants (split date ranges, locked in spec §3 and design §5):
    SPLIT_TRAIN_START, SPLIT_TRAIN_END
    SPLIT_VAL_START,   SPLIT_VAL_END
    SPLIT_TEST_START,  SPLIT_TEST_END
    WINSOR_QUANTILE

Design decisions (locked in design §5 and §9):
  - Split key is pl.col("t").dt.date() membership, NOT row index.
  - Three ranges are exhaustive and mutually exclusive.
  - Rows outside all three ranges receive None (split column = null).
  - Winsorization threshold is computed on train rows only (AC-WINSOR-1, AC-WINSOR-2).
  - Null delta_t_min rows are NOT clipped (AC-WINSOR-3).
  - Rows above threshold are clipped (not dropped) (AC-WINSOR-4).
  - Constants live here (not PRODUCTIVE_PARAMS) — evaluation protocol concern.
  - WINSOR_QUANTILE and split dates are not added to pyproject.toml.

Rolling origin
--------------
Every published result rests on ONE test window of 22 days (February 2024), so
nothing distinguishes "the method works" from "those 22 days happened to
cooperate" — the standard objection to a forecasting evaluation, and the one the
results document still declares open.

``ROLLING_FOLDS`` answers it by re-running the whole protocol at three origins,
each with its own 22-day test window. The window **expands** rather than slides:
every fold trains from the first day of data up to its own cutoff. That is the
usual "rolling origin with recalibration" design, and it has a second payoff
here — because the folds differ in training length as well as in period, a
result that holds across all three is robust to both.

The last fold is **exactly** the main split. That is deliberate: it makes the
published result the final origin of the sequence rather than a separate
analysis, and it means only two additional folds need training.

Fold boundaries are contiguous by construction: one fold's test window becomes
the next fold's validation window. No fold ever sees its own test period during
training, which is the only property that matters.
"""
from __future__ import annotations

from dataclasses import dataclass
from datetime import date, timedelta

import polars as pl

# ---------------------------------------------------------------------------
# Split date range constants (spec §3, inclusive on both ends)
# ---------------------------------------------------------------------------

SPLIT_TRAIN_START: date = date(2023, 10, 1)
SPLIT_TRAIN_END:   date = date(2024, 1, 15)

SPLIT_VAL_START:   date = date(2024, 1, 16)
SPLIT_VAL_END:     date = date(2024, 2, 7)

SPLIT_TEST_START:  date = date(2024, 2, 8)
SPLIT_TEST_END:    date = date(2024, 2, 29)

WINSOR_QUANTILE: float = 0.99


@dataclass(frozen=True)
class Fold:
    """One evaluation origin: three contiguous, ordered date ranges.

    Validated on construction. The invariants are not stylistic — a gap between
    ranges would silently drop days, and an overlap would train a model on its
    own test period, which is the failure this whole class exists to make
    impossible to introduce by hand.
    """

    name: str
    train_start: date
    train_end: date
    val_start: date
    val_end: date
    test_start: date
    test_end: date

    def __post_init__(self) -> None:
        for label, start, end in (
            ("train", self.train_start, self.train_end),
            ("val", self.val_start, self.val_end),
            ("test", self.test_start, self.test_end),
        ):
            if start > end:
                raise ValueError(
                    f"fold {self.name!r}: {label} range is inverted "
                    f"({start} > {end})"
                )
        day = timedelta(days=1)
        if self.val_start != self.train_end + day:
            raise ValueError(
                f"fold {self.name!r}: val must start the day after train ends; "
                f"train ends {self.train_end}, val starts {self.val_start}"
            )
        if self.test_start != self.val_end + day:
            raise ValueError(
                f"fold {self.name!r}: test must start the day after val ends; "
                f"val ends {self.val_end}, test starts {self.test_start}"
            )

    @property
    def train_days(self) -> int:
        return (self.train_end - self.train_start).days + 1

    @property
    def val_days(self) -> int:
        return (self.val_end - self.val_start).days + 1

    @property
    def test_days(self) -> int:
        return (self.test_end - self.test_start).days + 1

    def bounds(self) -> dict[str, tuple[date, date]]:
        """Split-name → (start, end), the shape the builders iterate."""
        return {
            "train": (self.train_start, self.train_end),
            "val": (self.val_start, self.val_end),
            "test": (self.test_start, self.test_end),
        }


#: The published split. Every frozen digest and every committed table is keyed
#: to it, so its dates must keep matching the module constants above.
MAIN_FOLD = Fold(
    name="main",
    train_start=SPLIT_TRAIN_START, train_end=SPLIT_TRAIN_END,
    val_start=SPLIT_VAL_START,     val_end=SPLIT_VAL_END,
    test_start=SPLIT_TEST_START,   test_end=SPLIT_TEST_END,
)

#: Three origins, oldest first. Each test window is 22 days, matching the main
#: split, so the folds differ in WHEN they are evaluated and in how much history
#: they were given — not in how much evidence each verdict rests on.
#:
#: Christmas and New Year land inside fold 1's test window and fold 2's
#: validation window. That is not a flaw to design around: if the result depends
#: on the holiday period, this is the analysis that has to reveal it.
ROLLING_FOLDS: tuple[Fold, ...] = (
    Fold(
        name="r1",
        train_start=date(2023, 10, 1), train_end=date(2023, 11, 30),
        val_start=date(2023, 12, 1),   val_end=date(2023, 12, 22),
        test_start=date(2023, 12, 23), test_end=date(2024, 1, 13),
    ),
    Fold(
        name="r2",
        train_start=date(2023, 10, 1), train_end=date(2023, 12, 22),
        val_start=date(2023, 12, 23),  val_end=date(2024, 1, 13),
        test_start=date(2024, 1, 14),  test_end=date(2024, 2, 4),
    ),
    MAIN_FOLD,
)


def fold_by_name(name: str) -> Fold:
    """Look up a fold by name, failing loudly on a typo.

    Builders take the fold as a string (CLI argument, notebook parameter), and a
    silent fallback to the main fold would produce results labelled as one origin
    and computed on another.
    """
    for fold in ROLLING_FOLDS:
        if fold.name == name:
            return fold
    known = ", ".join(fold.name for fold in ROLLING_FOLDS)
    raise KeyError(f"unknown fold {name!r}; known folds: {known}")


def split_temporal(df: pl.DataFrame, fold: Fold = MAIN_FOLD) -> pl.DataFrame:
    """Add a `split` column (Utf8) with values {"train", "val", "test"}.

    Membership is determined by pl.col("t").dt.date() against ``fold``'s six
    dates. Rows outside all three ranges receive null — expected for the rolling
    folds, whose windows end before the data does, and not expected for the main
    fold (the harness raises if found there).

    Parameters
    ----------
    df:
        headways DataFrame containing at least a `t` (Datetime) column.
    fold:
        Evaluation origin. Defaults to :data:`MAIN_FOLD`, so every existing
        caller keeps its exact behaviour and every frozen digest stays valid.

    Returns
    -------
    pl.DataFrame — input frame with one added column `split: Utf8`.
    """
    day = pl.col("t").dt.date()
    return df.with_columns(
        pl.when((day >= fold.train_start) & (day <= fold.train_end))
          .then(pl.lit("train"))
          .when((day >= fold.val_start) & (day <= fold.val_end))
          .then(pl.lit("val"))
          .when((day >= fold.test_start) & (day <= fold.test_end))
          .then(pl.lit("test"))
          .otherwise(None)
          .alias("split")
    )


def winsorize_train_p99(
    df: pl.DataFrame,
) -> tuple[pl.DataFrame, float]:
    """Clip delta_t_min to the 99th-percentile threshold computed on train rows only.

    The threshold is computed once as a scalar from non-null train-split rows.
    It is then applied as a clip ceiling to ALL rows (train + val + test).
    Null delta_t_min values are never clipped — they remain null (AC-WINSOR-3).

    Parameters
    ----------
    df:
        headways DataFrame that already has a `split` column (added by
        split_temporal) and a `delta_t_min` (Float64 nullable) column.

    Returns
    -------
    (clipped_df, threshold)
        clipped_df: same schema as df, delta_t_min clipped.
        threshold: the scalar train-p99 value used as the clip ceiling.

    Design note (AC-WINSOR-2 leakage guard):
        The filter `split == "train"` is applied BEFORE computing the quantile,
        so extreme outliers in val or test rows cannot shift the threshold.
    """
    threshold = float(
        df.filter(
            (pl.col("split") == "train") & pl.col("delta_t_min").is_not_null()
        )["delta_t_min"]
        .quantile(WINSOR_QUANTILE)
    )

    # Clip: preserve null rows; clip non-null rows to threshold from above.
    # pl.min_horizontal(col, lit(threshold)) would coerce null → 0 in some
    # polars versions, so we use the explicit when/then pattern (design §5).
    clipped = df.with_columns(
        pl.when(pl.col("delta_t_min").is_null())
          .then(None)
          .otherwise(
              pl.min_horizontal(pl.col("delta_t_min"), pl.lit(threshold))
          )
          .alias("delta_t_min")
    )
    return clipped, threshold

## Module: evaluation/metrics

In [ ]:
"""Evaluation metrics for headway forecasting — Fase 3.

Public API:
    mae(y_true, y_pred) -> float
    rmse(y_true, y_pred) -> float

Both functions accept polars Series (Float64) or numpy arrays (float64).
Null / NaN masking: rows where EITHER y_true or y_pred is null/NaN are
dropped before aggregation.  If no valid rows remain, ValueError is raised.

Design decisions locked in design §4:
  - ValueError on empty/all-null input (NOT silent NaN return).
  - Only MAE and RMSE are in scope (spec B3-NO-MAPE — ratio-based metrics
    are out of scope because near-zero headways cause denominator blow-up).
  - No new pyproject.toml dependencies (polars + numpy already present).
"""
from __future__ import annotations

import numpy as np
import polars as pl


def _to_numpy_with_mask(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Coerce both inputs to float64 numpy arrays and apply the null/NaN mask.

    Polars Series with dtype Float64: null cells become NaN via .to_numpy().
    numpy arrays: assumed to already use NaN for missing values.

    Returns
    -------
    (y_true_masked, y_pred_masked) — two 1-D float64 arrays of equal length,
    containing no NaN values.  May be empty if all rows were masked.
    """
    # Coerce to numpy.
    if isinstance(y_true, pl.Series):
        yt = y_true.to_numpy(allow_copy=True).astype(np.float64)
    else:
        yt = np.asarray(y_true, dtype=np.float64).ravel()

    if isinstance(y_pred, pl.Series):
        yp = y_pred.to_numpy(allow_copy=True).astype(np.float64)
    else:
        yp = np.asarray(y_pred, dtype=np.float64).ravel()

    # Elementwise mask: keep row only if BOTH sides are finite (not NaN).
    mask = ~(np.isnan(yt) | np.isnan(yp))
    return yt[mask], yp[mask]


def mae(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> float:
    """Mean Absolute Error in minutes, with null/NaN masking.

    Parameters
    ----------
    y_true, y_pred:
        Ground-truth and predicted headway values in minutes.
        Accepts polars Series (Float64) or numpy arrays (float64).
        Null / NaN positions in either input are dropped before computation.

    Returns
    -------
    float — MAE in minutes.

    Raises
    ------
    ValueError
        If the masked input is empty (all-null or zero-length).
    """
    yt, yp = _to_numpy_with_mask(y_true, y_pred)
    if len(yt) == 0:
        raise ValueError(
            "mae: metric on empty/all-null input — no valid (y_true, y_pred) pairs."
        )
    return float(np.mean(np.abs(yt - yp)))


def rmse(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> float:
    """Root Mean Squared Error in minutes, with null/NaN masking.

    Parameters
    ----------
    y_true, y_pred:
        Ground-truth and predicted headway values in minutes.
        Accepts polars Series (Float64) or numpy arrays (float64).
        Null / NaN positions in either input are dropped before computation.

    Returns
    -------
    float — RMSE in minutes.

    Raises
    ------
    ValueError
        If the masked input is empty (all-null or zero-length).
    """
    yt, yp = _to_numpy_with_mask(y_true, y_pred)
    if len(yt) == 0:
        raise ValueError(
            "rmse: metric on empty/all-null input — no valid (y_true, y_pred) pairs."
        )
    return float(np.sqrt(np.mean((yt - yp) ** 2)))

## Module: data/windowing

Embebido solo por `compute_max_N`. `make_window_index` NO se usa acá.

In [ ]:
"""Windowing module for supervised dataset construction — Fase 3 DL.

AC-WIN-1: Build window index per slot.
AC-WIN-2: Stride-parametrized index generation.
AC-WIN-3: Deterministic slot-boundary-respecting index.
AC-WIN-4: Exported constants DEFAULT_T_IN, DEFAULT_T_OUT, DEFAULT_STRIDE.
AC-WIN-5: Empty-slot guard (returns zero entries when N < T_in + T_out).
AC-WIN-6: Zero torch imports at module level.
AC-MAXN-1: compute_max_N returns train-p99 of (n_buses-1) per (empresaid, direction).
AC-MAXN-2: compute_max_N is called on train-only df; leakage is caller responsibility.

Design decisions (locked in design §2.2 and §5):
  - WindowIndexEntry: TypedDict with empresaid, direction, pair_rank, start_idx.
  - start_idx is relative to the sorted slot frame (not the full df).
  - Slot key: (empresaid, direction, pair_rank).
  - No torch imports anywhere in this module (INV-10, DL-10).
"""
from __future__ import annotations

import math
from typing import TypedDict

import polars as pl

# ---------------------------------------------------------------------------
# Constants (locked in design §5 — DL-1)
# ---------------------------------------------------------------------------

DEFAULT_T_IN: int = 12
DEFAULT_T_OUT: int = 1
DEFAULT_STRIDE: int = 1

_SLOT_COLS: list[str] = ["empresaid", "direction", "pair_rank"]


class WindowIndexEntry(TypedDict):
    """Single window anchor.

    empresaid: int — corridor identifier.
    direction: int — bus direction (-1 or +1).
    pair_rank: int — positional slot index within a snapshot.
    start_idx: int — row index into the sorted slot frame where this window starts.
                     The window covers rows [start_idx, start_idx + T_in + T_out).
    """

    empresaid: int
    direction: int
    pair_rank: int
    start_idx: int


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _slot_lengths(df: pl.DataFrame) -> pl.DataFrame:
    """Return a DataFrame with (empresaid, direction, pair_rank, n_rows).

    Used by make_window_index to determine how many windows each slot produces.
    The count is over ALL rows (null delta_t_min counts — windowing does not
    drop null rows; the Dataset layer handles null masking later).
    """
    return (
        df.group_by(_SLOT_COLS)
        .agg(pl.len().alias("n_rows"))
    )


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def compute_max_N(
    train_df: pl.DataFrame,
    *,
    quantile: float = 0.99,
) -> dict[tuple[int, int], int]:
    """Train-p99 of (n_buses - 1) per (empresaid, direction). DL-5. AC-MAXN-1..2.

    Parameters
    ----------
    train_df:
        DataFrame filtered to train rows only (caller responsibility).
        Must have columns: empresaid (Int64), direction (Int64), n_buses (Int32).
    quantile:
        Percentile for the cap (default 0.99 per DL-5).

    Returns
    -------
    dict[(empresaid, direction), int] — the maximum slot index (0-based max_N).
    Returned values are Python int (not np.int64) so they can be used as tensor
    dimensions directly.
    """
    # Compute quantile of (n_buses - 1) per (empresaid, direction).
    # We use unique snapshots: each row in the windowing context represents one
    # (empresaid, direction, snapshot) combination. n_buses is per snapshot.
    result: dict[tuple[int, int], int] = {}

    # Group by (empresaid, direction) and compute the p99 of (n_buses - 1).
    stats = (
        train_df
        .with_columns(
            (pl.col("n_buses") - 1).alias("_n_slots")
        )
        .group_by(["empresaid", "direction"])
        .agg(
            pl.col("_n_slots").quantile(quantile).alias("max_N_float")
        )
    )

    for row in stats.iter_rows(named=True):
        key = (int(row["empresaid"]), int(row["direction"]))
        result[key] = int(math.floor(row["max_N_float"]))

    return result


def make_window_index(
    df: pl.DataFrame,
    *,
    T_in: int = DEFAULT_T_IN,
    T_out: int = DEFAULT_T_OUT,
    horizon: int | None = None,
    stride: int = DEFAULT_STRIDE,
) -> list[WindowIndexEntry]:
    """Deterministic per-slot window index. DL-1, DL-11. AC-WIN-1..5, AC-WIN-H1..H3.

    Produces a list of WindowIndexEntry dicts where each entry anchors one
    training window. Entries are sorted by (empresaid, direction, pair_rank,
    start_idx) for determinism.

    Parameters
    ----------
    df:
        headways DataFrame sorted (or sortable) by (slot_cols, t).
        Columns required: empresaid, direction, pair_rank, t.
    T_in:
        Input sequence length (number of timesteps fed to model).
    T_out:
        Prediction sequence length (number of future timesteps). Retained for
        backward compatibility. Default 1.
    horizon:
        DIRECT-horizon prediction offset. When provided, ``window_size = T_in + horizon``
        (overrides the T_out contribution). Default ``None`` falls back to T_out semantics
        so existing callers are unaffected. ``horizon=1`` produces results identical to
        ``T_out=1`` (AC-WIN-H3).
    stride:
        Step between consecutive window starts (default 1 = every timestep).

    Returns
    -------
    list[WindowIndexEntry] — may be empty if no slot has enough rows.
    """
    window_size = T_in + (horizon if horizon is not None else T_out)
    index: list[WindowIndexEntry] = []

    # Partition by slot to keep slot boundaries clean (AC-WIN-3).
    slots = df.sort(_SLOT_COLS + ["t"]).partition_by(_SLOT_COLS, maintain_order=True)

    for slot_df in slots:
        if slot_df.is_empty():
            continue

        n_rows = len(slot_df)
        if n_rows < window_size:
            # AC-WIN-5: not enough rows for even one window — skip.
            continue

        # Extract slot key from first row.
        first = slot_df.row(0, named=True)
        emp: int = int(first["empresaid"])
        direction: int = int(first["direction"])
        pr: int = int(first["pair_rank"])

        # Generate start indices with stride.
        # Number of valid windows: floor((n_rows - window_size) / stride) + 1
        n_windows = math.floor((n_rows - window_size) / stride) + 1
        for w in range(n_windows):
            start_idx = w * stride
            index.append(
                WindowIndexEntry(
                    empresaid=emp,
                    direction=direction,
                    pair_rank=pr,
                    start_idx=start_idx,
                )
            )

    return index

## Module: data/sample_index  (contratos C1 + C2)

In [ ]:
"""Canonical sample index — the shared population contract (C1 + C2).

This module exists because the project never wrote down what a sample *is*.
``windowing.make_window_index`` anchors windows on a **row index** inside a
``(empresaid, direction, pair_rank)`` slot, which produces two defects:

  C1 violation — the target of a snapshot is emitted once per anchoring slot,
  so every target is counted 2.4-5.4 times and the reported MAE is a
  fleet-density-weighted mean.

  C2 violation — consecutive row positions are not checked to be consecutive
  minutes, so the nominal horizon is a row offset, not a time offset. A window
  crossing a day boundary or a trip cut yields a "10-minute" target that is
  hours away.

``windowing.make_window_index`` is deliberately left untouched: notebooks 12/13
(and the E4 twins 18/19) must keep reproducing the frozen architecture
comparison, whose validity rests on all three architectures sharing the same
flaw. This module is the population for the *retrained* pipeline only.

Contract enforced here
----------------------
C1  A sample is ``(empresaid, direction, start_ts, horizon)``. The anchor is an
    instant, not a row position, and each one is emitted exactly once.
C2  A sample is valid only when the ``T_in + horizon`` snapshot timestamps
    starting at ``start_ts`` are strictly consecutive minutes, so the target
    lands exactly ``horizon`` minutes after the end of the input window.

``pair_rank`` survives as the position *within* the predicted vector, never as
an anchoring axis.
"""
from __future__ import annotations

from typing import TypedDict

import numpy as np
import polars as pl

# One snapshot per minute: the grid the headway series is resampled onto.
GRID_STEP_MINUTES: int = 1

# Columns that identify one snapshot series. `pair_rank` is intentionally absent.
_SERIES_COLS: list[str] = ["empresaid", "direction"]


class SampleIndexEntry(TypedDict):
    """One canonical sample.

    empresaid: int — corridor identifier.
    direction: int — bus direction (-1 or +1).
    start_ts: the timestamp of the FIRST input snapshot.
    target_ts: the timestamp of the predicted snapshot. Always exactly
               ``horizon`` minutes after the last input snapshot.
    horizon: int — prediction offset in minutes (now genuinely minutes).
    """

    empresaid: int
    direction: int
    start_ts: object
    target_ts: object
    horizon: int


def _contiguous_run_mask(ts: np.ndarray, span: int) -> np.ndarray:
    """Mask of window starts whose ``span`` timestamps are consecutive minutes.

    Parameters
    ----------
    ts:
        Sorted, unique timestamps for one series, as numpy datetime64.
    span:
        Number of consecutive timestamps a valid window must cover
        (``T_in + horizon``).

    Returns
    -------
    Boolean array of length ``len(ts)``. ``mask[i]`` is True when
    ``ts[i:i+span]`` are consecutive minutes. Positions with fewer than
    ``span`` timestamps remaining are False.
    """
    n = ts.size
    mask = np.zeros(n, dtype=bool)
    if n < span:
        return mask

    step = np.timedelta64(GRID_STEP_MINUTES, "m")
    # gap[i] is True when ts[i+1] follows ts[i] by exactly one grid step.
    gap_ok = np.diff(ts) == step

    # A window at i needs the span-1 gaps starting at i to all be contiguous.
    # Cumulative-sum trick: count of good gaps in [i, i+span-2] must equal span-1.
    need = span - 1
    if need == 0:
        mask[:] = True
        return mask

    cum = np.concatenate(([0], np.cumsum(gap_ok)))
    starts = np.arange(n - span + 1)
    good = cum[starts + need] - cum[starts]
    mask[: n - span + 1] = good == need
    return mask


def make_sample_index(
    df: pl.DataFrame,
    *,
    horizon: int,
    T_in: int,
) -> pl.DataFrame:
    """Canonical sample index for one frame. Enforces C1 and C2.

    Parameters
    ----------
    df:
        Headway frame. Required columns: ``empresaid``, ``direction``, ``t``.
        ``pair_rank`` may be present; it is ignored for anchoring.
    horizon:
        Prediction offset in minutes.
    T_in:
        Input window length in snapshots.

    Returns
    -------
    DataFrame with one row per canonical sample, columns
    ``empresaid, direction, start_ts, target_ts, horizon``, sorted
    deterministically. Empty (with the right schema) when no window qualifies.
    """
    if horizon < 1:
        raise ValueError(f"horizon must be >= 1, got {horizon}")
    if T_in < 1:
        raise ValueError(f"T_in must be >= 1, got {T_in}")

    span = T_in + horizon
    rows: list[pl.DataFrame] = []

    # One series per (empresaid, direction) — NOT per pair_rank. That collapse is
    # the whole point of C1: the target is the snapshot, not the slot.
    series = (
        df.select(_SERIES_COLS + ["t"])
        .unique()
        .sort(_SERIES_COLS + ["t"])
        .partition_by(_SERIES_COLS, maintain_order=True)
    )

    for s in series:
        if s.is_empty():
            continue
        ts = s.get_column("t").to_numpy()
        mask = _contiguous_run_mask(ts, span)
        if not mask.any():
            continue

        first = s.row(0, named=True)
        starts = ts[mask]
        # The target sits `horizon` minutes after the LAST input snapshot, which
        # is start + (T_in - 1) steps. Contiguity is already guaranteed by mask.
        targets = starts + np.timedelta64((T_in - 1 + horizon) * GRID_STEP_MINUTES, "m")

        rows.append(
            pl.DataFrame(
                {
                    "empresaid": np.full(starts.size, int(first["empresaid"]), dtype=np.int64),
                    "direction": np.full(starts.size, int(first["direction"]), dtype=np.int64),
                    "start_ts": starts,
                    "target_ts": targets,
                    "horizon": np.full(starts.size, int(horizon), dtype=np.int64),
                }
            )
        )

    if not rows:
        return pl.DataFrame(
            schema={
                "empresaid": pl.Int64,
                "direction": pl.Int64,
                "start_ts": df.schema["t"],
                "target_ts": df.schema["t"],
                "horizon": pl.Int64,
            }
        )

    return pl.concat(rows).sort(["empresaid", "direction", "start_ts"])


def effective_horizon_minutes(
    index: pl.DataFrame,
    *,
    T_in: int,
) -> pl.Series:
    """Realized gap in minutes between end-of-window and target, per sample.

    Under C2 this is constant and equal to ``horizon`` for every row. It is
    exposed so a test can assert that rather than trust the construction.
    """
    window_end = pl.col("start_ts") + pl.duration(minutes=(T_in - 1) * GRID_STEP_MINUTES)
    return (
        index.select(
            ((pl.col("target_ts") - window_end).dt.total_minutes()).alias("eff")
        )
        .get_column("eff")
    )

## Module: data/context_features

Embebido porque `fitted.py` lo referencia. La bandera atípica **no** se usa.

In [ ]:
"""Context features module for supervised dataset construction — Fase 3 DL.

AC-CTX-1: encode_context adds hour_sin, hour_cos at midnight → (0, 1).
AC-CTX-2: encode_context adds dow_sin, dow_cos with period 7; emits 5 named columns.
AC-CTX-3: load_atypical_days(None) returns empty set (graceful fallback, DL-2).
AC-CTX-4: load_atypical_days(path) returns set[date] from CSV when file exists.
AC-CTX-5: atypical_flag=1.0 when timestamp date in atypical_dates, else 0.0.
AC-CTX-6: zero torch imports at module level (INV-10, DL-10).

Design decisions locked in design §2.4 and §5:
  - encode_context operates on a DataFrame with a `t` (Datetime) column.
  - Cyclical encoding: sin(2π * value / period), cos(2π * value / period).
  - atypical_flag = 1.0 when t.date() in atypical_dates else 0.0.
  - DL-2: graceful fallback to atypical_flag=0 when path is None or missing.
  - No torch imports (INV-10).
"""
from __future__ import annotations

import logging
import math
import warnings
from datetime import date
from pathlib import Path

import polars as pl

_log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

CONTEXT_FEATURE_NAMES: tuple[str, ...] = (
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "atypical_flag",
)


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _cyclical_pair(col: pl.Expr, period: int, prefix: str) -> list[pl.Expr]:
    """Emit [sin_expr, cos_expr] aliased <prefix>_sin, <prefix>_cos.

    Encoding: sin(2π * col / period), cos(2π * col / period).

    Parameters
    ----------
    col:
        Polars expression that yields a numeric value (e.g. hour 0-23, dow 0-6).
    period:
        Full cycle length (24 for hour, 7 for day-of-week).
    prefix:
        Column name prefix ("hour" or "dow").
    """
    angle = col * (2.0 * math.pi / period)
    return [
        angle.sin().alias(f"{prefix}_sin"),
        angle.cos().alias(f"{prefix}_cos"),
    ]


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def encode_context(
    df: pl.DataFrame,
    *,
    atypical_dates: set[date] | None = None,
) -> pl.DataFrame:
    """Add 5 context columns derived from the `t` (Datetime) column.

    AC-CTX-1..5. DL-2 graceful fallback: atypical_flag=0 when atypical_dates
    is None or empty.

    Parameters
    ----------
    df:
        DataFrame with a `t` (Datetime[us]) column.
    atypical_dates:
        Set of dates that are atypical (e.g. holidays, strikes). When None
        or empty, atypical_flag is 0.0 for all rows.

    Returns
    -------
    pl.DataFrame — input frame with 5 additional columns appended in the order
    defined by CONTEXT_FEATURE_NAMES.
    """
    if atypical_dates is None:
        atypical_dates = set()

    # Cyclical hour and day-of-week encodings.
    # polars dt.weekday() returns ISO weekday: Monday=1 .. Sunday=7.
    # We convert to 0-indexed (Monday=0 .. Sunday=6) to align with Python convention
    # so that midnight Monday → dow=0 → dow_sin=sin(0)=0, dow_cos=cos(0)=1 (AC-CTX-1).
    hour_expr = pl.col("t").dt.hour().cast(pl.Float64)
    dow_expr = (pl.col("t").dt.weekday() - 1).cast(pl.Float64)

    sin_cos_exprs: list[pl.Expr] = [
        *_cyclical_pair(hour_expr, 24, "hour"),
        *_cyclical_pair(dow_expr, 7, "dow"),
    ]

    # Atypical flag: 1.0 if the date is in the atypical set, else 0.0.
    if atypical_dates:
        # Build a list of date literals to check membership against.
        atypical_list = sorted(atypical_dates)
        date_col = pl.col("t").dt.date()
        flag_expr = pl.lit(0.0)

        # Chain when/then for each atypical date.
        flag_chain = pl.when(
            date_col == pl.lit(atypical_list[0])
        ).then(pl.lit(1.0))
        for d in atypical_list[1:]:
            flag_chain = flag_chain.when(
                date_col == pl.lit(d)
            ).then(pl.lit(1.0))
        flag_expr = flag_chain.otherwise(pl.lit(0.0))
    else:
        flag_expr = pl.lit(0.0)

    return df.with_columns(
        *sin_cos_exprs,
        flag_expr.cast(pl.Float64).alias("atypical_flag"),
    )


def load_atypical_days(
    path: Path | str | None,
) -> set[date]:
    """Read CSV with at least a `date` column; return set[date].

    AC-CTX-3 + DL-2: returns empty set when path is None OR file does not exist.
    A warning is emitted when the path is non-None but missing (so callers know
    the fallback was triggered — not a silent failure).

    Parameters
    ----------
    path:
        Path to a CSV file with a `date` column (ISO-8601 format).
        May be None, a string, or a Path object.

    Returns
    -------
    set[date] — parsed dates, or empty set on fallback.
    """
    if path is None:
        return set()

    resolved = Path(path)
    if not resolved.exists():
        warnings.warn(
            f"load_atypical_days: file not found at '{resolved}'; "
            "falling back to empty atypical set (atypical_flag=0 for all rows). "
            "DL-2 graceful fallback.",
            stacklevel=2,
        )
        return set()

    df = pl.read_csv(resolved, try_parse_dates=True)
    # The frozen 02-eda-corridors CSV names its date column `day`; older
    # fixtures use `date`. Accept either, preferring `date` when both exist.
    date_col = next((c for c in ("date", "day") if c in df.columns), None)
    if date_col is None:
        warnings.warn(
            f"load_atypical_days: CSV at '{resolved}' has no 'date' or 'day' column; "
            "falling back to empty set.",
            stacklevel=2,
        )
        return set()

    dates: set[date] = set()
    for val in df[date_col].to_list():
        if val is not None:
            if isinstance(val, date):
                dates.add(val)
            else:
                try:
                    from datetime import datetime as _dt
                    dates.add(_dt.fromisoformat(str(val)).date())
                except ValueError:
                    _log.warning("Skipping unparseable date value: %s", val)

    return dates

## Module: baselines/fitted

⚠️ Embebido **solo** por `sample_search_configs`, `SEARCH_SPACE`, `SEARCH_SEED` y `SEARCH_N_CONFIGS`. Su `_build_features` arrastra el defecto de lags posicionales y **no se llama** en este notebook.

In [ ]:
"""Fitted ML baseline for headway forecasting — gradient-boosted regressor (B5_XGB).

Why this module is separate from `statistical.py`:
    B0-B4 are closed-form/recursive predictors with NO learned parameters and a
    "no new dependencies" design lock. B5_XGB is a *fitted* learner (XGBoost) —
    a different category. It answers the reviewer reflex "where is a fitted/ML
    baseline?" that pure naive baselines (persistence, moving average, SES,
    historical average) do not.

Design — fair comparison to the DL models (NB11-13, NB17-19):
    The DL models consume an input window of T_in = 12 consecutive 1-minute
    steps and predict the headway HORIZON steps after the last input step. The
    XGBoost baseline is given the SAME information: 12 lagged headway values
    ending HORIZON steps before the target, so `lag_1` equals the B1 persistence
    prediction (`shift(horizon)`) and the model strictly extends the naive
    baselines rather than seeing extra future data. Calendar context (hour,
    weekday), static slot keys (direction, pair_rank) and the atypical-day flag
    round out the features.

    Two asymmetries versus the DL models were removed (peer-review fix):
      1. ATYPICAL-DAY FLAG. The DL models receive `atypical_flag` as a required,
         hash-pinned context feature. B5_XGB now receives the same binary flag,
         built with the SAME `encode_context` helper so the semantics cannot
         drift between the two model families.
      2. HYPERPARAMETER SEARCH. The DL models were tuned; B5_XGB used a single
         hardcoded configuration. It now runs a seeded random search of
         `SEARCH_N_CONFIGS` configurations selected STRICTLY on the validation
         split (see `_random_search`), with the winning configuration reported
         back to the caller so it can be persisted and audited.

Contract (mirrors statistical.py):
    predict_b5_xgb(headways, *, horizon=1, seed=42, atypical_dates=None,
                   search=True) -> headways + y_pred_b5_xgb
    fit_predict_b5_xgb(...) -> B5FitResult (predictions + search provenance)

    Input must have the `split` column (added by split_temporal). The model is
    fit on TRAIN rows only; predictions are produced for ALL rows. Validation
    rows are used for hyperparameter selection and early stopping ONLY when
    there are enough of them (>= _MIN_VAL_ROWS); otherwise the frozen default
    configuration and a fixed number of trees are used.

Leakage contract (hard):
    The `test` split NEVER influences training, early stopping, or
    hyperparameter selection. Selection reads the validation loss only.

Atypical-flag contract (mirrors the DL notebooks):
    `atypical_dates=None` means "no atypical calendar supplied" (library/fixture
    use) and yields an all-zero flag column. Passing an EXPLICIT EMPTY SET is a
    configuration error — a CSV that parsed to nothing must fail closed instead
    of silently disabling the feature — and raises ValueError.

Determinism:
    Fixed `seed`, fixed `SEARCH_SEED` for the configuration sampler, and
    `tree_method="hist"` with a pinned `nthread` → repeated calls on the same
    machine with the same inputs produce identical predictions and select the
    same configuration.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import date

import numpy as np
import polars as pl


_SLOT_COLS: list[str] = ["empresaid", "direction", "pair_rank"]

# Number of lagged headway steps fed to the model = DL input window (T_in).
N_LAGS: int = 12

# Use validation rows for search + early stopping only when there are at least
# this many; tiny test fixtures (and corridors with no val rows) fall back to
# the frozen default configuration and a fixed number of trees.
_MIN_VAL_ROWS: int = 50

# Threads for the Kaggle CPU kernel (4 vCPU). Pinned in source: XGBoost `hist`
# is reproducible for a FIXED thread count, so this value is part of the
# determinism contract and must not be made environment-dependent.
_NTHREAD: int = 4

_NUM_BOOST_ROUND: int = 400
_EARLY_STOPPING_ROUNDS: int = 30

# Cheaper budget for the search sweep; the winner is refit at the full budget.
_SEARCH_NUM_BOOST_ROUND: int = 200
_SEARCH_EARLY_STOPPING_ROUNDS: int = 20

# Frozen fallback configuration (the pre-search hardcoded baseline). Used when
# there is no usable validation split, or when `search=False`.
_XGB_PARAMS: dict = {
    "eta": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "lambda": 1.0,
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "nthread": _NTHREAD,
}

# ---------------------------------------------------------------------------
# Hyperparameter random search — validation-only selection.
# ---------------------------------------------------------------------------

# EXACTLY 24 configurations: the agreed budget for a Kaggle CPU kernel that
# must fit 2 corridors x 4 horizons within the session runtime limit.
SEARCH_N_CONFIGS: int = 24

# Fixed in source so the search is reproducible and cannot be silently
# re-rolled between runs. Changing this value changes the paper's numbers.
SEARCH_SEED: int = 20240718

# Discrete search space (|space| = 6*6*5*5*5*5 = 22500 >> 24 draws).
SEARCH_SPACE: dict[str, list] = {
    "eta": [0.02, 0.03, 0.05, 0.08, 0.12, 0.20],
    "max_depth": [3, 4, 5, 6, 8, 10],
    "min_child_weight": [1, 3, 5, 10, 20],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "lambda": [0.5, 1.0, 2.0, 5.0, 10.0],
}


def sample_search_configs(
    n_configs: int = SEARCH_N_CONFIGS, *, seed: int = SEARCH_SEED
) -> list[dict]:
    """Draw `n_configs` DISTINCT hyperparameter configurations, deterministically.

    Sampling depends only on `seed` and `SEARCH_SPACE` — never on the data — so
    every corridor and horizon evaluates the same candidate set and the sweep is
    exactly reproducible.

    Returns
    -------
    list[dict] — each dict holds only the searched keys (eta, max_depth,
    min_child_weight, subsample, colsample_bytree, lambda).
    """
    rng = np.random.default_rng(seed)
    keys = sorted(SEARCH_SPACE)  # sorted → draw order independent of dict order
    seen: set[tuple] = set()
    configs: list[dict] = []
    # Bounded loop: the space is ~22500 wide, so 24 distinct draws are reached
    # almost immediately; the cap only guards against a shrunken space.
    for _ in range(n_configs * 1000):
        if len(configs) == n_configs:
            break
        values = tuple(
            SEARCH_SPACE[k][int(rng.integers(len(SEARCH_SPACE[k])))] for k in keys
        )
        if values in seen:
            continue
        seen.add(values)
        configs.append(dict(zip(keys, values)))
    if len(configs) != n_configs:
        raise ValueError(
            f"sample_search_configs: could only draw {len(configs)} distinct "
            f"configurations out of {n_configs} requested"
        )
    return configs


@dataclass(frozen=True)
class B5FitResult:
    """Predictions plus the provenance needed to audit the fitted baseline.

    Attributes
    ----------
    predictions:
        Input frame (sorted by slot, t) with `y_pred_b5_xgb` added.
    best_params:
        The full parameter dict handed to XGBoost for the final fit.
    best_val_rmse:
        Validation RMSE of the selected configuration (``nan`` when no search
        ran, i.e. no usable validation split).
    best_iteration:
        Boosting iteration chosen by early stopping (-1 when unavailable).
    n_configs_evaluated:
        How many configurations the search actually fit (0 when it was skipped).
    search_seed:
        Seed used to draw the candidate configurations.
    used_atypical_flag:
        True when a non-empty atypical calendar was supplied.
    """

    predictions: pl.DataFrame
    best_params: dict = field(default_factory=dict)
    best_val_rmse: float = float("nan")
    best_iteration: int = -1
    n_configs_evaluated: int = 0
    search_seed: int = SEARCH_SEED
    used_atypical_flag: bool = False


def _build_features(
    headways: pl.DataFrame,
    *,
    horizon: int,
    atypical_dates: set[date] | None = None,
) -> tuple[pl.DataFrame, list[str]]:
    """Return the frame sorted by (slot, t) with lag + context feature columns
    added, plus the list of feature column names.

    lag_k (k = 1..N_LAGS) = headway value (forward-filled within slot) observed
    `horizon + k - 1` steps before the target row. lag_1 == B1 persistence.

    `_atypical` is the SAME binary flag the DL models consume: it is produced by
    `encode_context`, not reimplemented here, so the two model families cannot
    diverge on what counts as an atypical day.

    Raises
    ------
    ValueError
        If `atypical_dates` is an explicit empty set (fail closed — see the
        module docstring's atypical-flag contract).
    """
    if atypical_dates is not None and len(atypical_dates) == 0:
        raise ValueError(
            "_build_features: atypical_dates parsed to an EMPTY set. The "
            "atypical-day feature must not be silently disabled — pass None "
            "only when no atypical calendar exists at all."
        )

    lag_exprs = [
        pl.col("delta_t_min")
        .forward_fill()
        .shift(horizon + k - 1)
        .over(_SLOT_COLS)
        .alias(f"_lag_{k}")
        for k in range(1, N_LAGS + 1)
    ]
    df = (
        encode_context(headways, atypical_dates=atypical_dates)
        .sort(_SLOT_COLS + ["t"])
        .with_columns(
            *lag_exprs,
            pl.col("t").dt.hour().alias("_hour"),
            pl.col("t").dt.weekday().alias("_weekday"),
            pl.col("atypical_flag").alias("_atypical"),
        )
    )
    feature_cols = (
        [f"_lag_{k}" for k in range(1, N_LAGS + 1)]
        + ["_hour", "_weekday", "direction", "pair_rank", "_atypical"]
    )
    return df, feature_cols


def _random_search(
    xgb,
    dtrain,
    dval,
    *,
    seed: int,
    n_configs: int,
    search_seed: int,
) -> tuple[dict, float, int, int]:
    """Fit `n_configs` candidates and keep the one with the lowest VALIDATION RMSE.

    The only signal read here is `booster.best_score` on `dval` — the test split
    is not part of either DMatrix, so selection cannot see it.

    Returns
    -------
    (best_params, best_val_rmse, best_iteration, n_evaluated)
    """
    best_params: dict = {}
    best_score = float("inf")
    best_iteration = -1
    n_evaluated = 0

    for candidate in sample_search_configs(n_configs, seed=search_seed):
        params = dict(_XGB_PARAMS, **candidate, seed=seed)
        booster = xgb.train(
            params,
            dtrain,
            num_boost_round=_SEARCH_NUM_BOOST_ROUND,
            evals=[(dval, "val")],
            early_stopping_rounds=_SEARCH_EARLY_STOPPING_ROUNDS,
            verbose_eval=False,
        )
        n_evaluated += 1
        score = float(booster.best_score)
        # Strict `<` → first-drawn config wins ties, keeping selection
        # deterministic for a fixed candidate order.
        if score < best_score:
            best_score = score
            best_params = params
            best_iteration = int(getattr(booster, "best_iteration", -1))

    return best_params, best_score, best_iteration, n_evaluated


def fit_predict_b5_xgb(
    headways: pl.DataFrame,
    *,
    horizon: int = 1,
    seed: int = 42,
    atypical_dates: set[date] | None = None,
    search: bool = True,
    n_configs: int = SEARCH_N_CONFIGS,
    search_seed: int = SEARCH_SEED,
) -> B5FitResult:
    """Fit B5_XGB and return predictions plus the auditable search provenance.

    Parameters
    ----------
    headways:
        headways DataFrame with the `split` column attached. Columns consumed:
        empresaid, t, direction, pair_rank, delta_t_min, split.
    horizon:
        Forecast horizon in steps. lag_1 = shift(horizon) so the 1-lag feature
        equals B1 persistence; horizon=1 is the default.
    seed:
        XGBoost training seed (reproducible tree construction).
    atypical_dates:
        Atypical-day calendar (the same set the DL models receive). None means
        "not supplied" → all-zero flag; an explicit empty set raises.
    search:
        When True (default) run the seeded validation-only random search. When
        False, or when there is no usable validation split, the frozen default
        configuration is used.
    n_configs / search_seed:
        Search budget and sampler seed. Both default to the frozen constants;
        overriding them is a test/debug affordance, not a production path.

    Returns
    -------
    B5FitResult
    """
    import xgboost as xgb

    original_cols = headways.columns
    df, feature_cols = _build_features(
        headways, horizon=horizon, atypical_dates=atypical_dates
    )
    used_atypical = bool(atypical_dates)

    is_train = df["split"] == "train"
    is_val = df["split"] == "val"
    target_present = df["delta_t_min"].is_not_null()

    train_mask = (is_train & target_present).to_numpy()
    n_train = int(train_mask.sum())

    # Degenerate: nothing to fit on → null predictions (mirrors B0 on empty slots).
    if n_train == 0:
        return B5FitResult(
            predictions=df.select(original_cols).with_columns(
                pl.lit(None, dtype=pl.Float64).alias("y_pred_b5_xgb")
            ),
            used_atypical_flag=used_atypical,
        )

    X_all = df.select(feature_cols).to_numpy().astype(np.float64)
    y_all = df["delta_t_min"].to_numpy().astype(np.float64)

    dtrain = xgb.DMatrix(X_all[train_mask], label=y_all[train_mask], missing=np.nan)
    dall = xgb.DMatrix(X_all, missing=np.nan)

    val_mask = (is_val & target_present).to_numpy()
    has_val = int(val_mask.sum()) >= _MIN_VAL_ROWS

    params = dict(_XGB_PARAMS, seed=seed)
    best_val_rmse = float("nan")
    best_iteration = -1
    n_evaluated = 0

    if has_val:
        # NOTE: dval holds VALIDATION rows only. The test split is absent from
        # every DMatrix built here, so neither early stopping nor hyperparameter
        # selection can read it.
        dval = xgb.DMatrix(X_all[val_mask], label=y_all[val_mask], missing=np.nan)
        if search:
            params, best_val_rmse, _search_iter, n_evaluated = _random_search(
                xgb,
                dtrain,
                dval,
                seed=seed,
                n_configs=n_configs,
                search_seed=search_seed,
            )
        # Refit the selected configuration at the full boosting budget, keeping
        # the existing early-stopping-on-validation behaviour.
        booster = xgb.train(
            params,
            dtrain,
            num_boost_round=_NUM_BOOST_ROUND,
            evals=[(dval, "val")],
            early_stopping_rounds=_EARLY_STOPPING_ROUNDS,
            verbose_eval=False,
        )
        best_val_rmse = float(booster.best_score)
        best_iteration = int(getattr(booster, "best_iteration", -1))
    else:
        booster = xgb.train(params, dtrain, num_boost_round=_NUM_BOOST_ROUND)

    preds = booster.predict(dall).astype(np.float64)

    return B5FitResult(
        predictions=df.select(original_cols).with_columns(
            pl.Series("y_pred_b5_xgb", preds, dtype=pl.Float64)
        ),
        best_params=dict(params),
        best_val_rmse=best_val_rmse,
        best_iteration=best_iteration,
        n_configs_evaluated=n_evaluated,
        search_seed=search_seed,
        used_atypical_flag=used_atypical,
    )


def predict_b5_xgb(
    headways: pl.DataFrame,
    *,
    horizon: int = 1,
    seed: int = 42,
    atypical_dates: set[date] | None = None,
    search: bool = True,
) -> pl.DataFrame:
    """Add column `y_pred_b5_xgb`: gradient-boosted forecast of delta_t_min.

    Thin wrapper over :func:`fit_predict_b5_xgb` for callers that only need the
    predictions. See that function for the full parameter documentation.

    Returns
    -------
    pl.DataFrame — input frame (sorted by slot, t) with `y_pred_b5_xgb`
        (Float64 nullable) added. If the train split has no usable rows, the
        column is all-null.
    """
    return fit_predict_b5_xgb(
        headways,
        horizon=horizon,
        seed=seed,
        atypical_dates=atypical_dates,
        search=search,
    ).predictions

## Module: baselines/contiguous_features

`build_contiguous_features` — lags leídos de la ventana contigua.

In [ ]:
"""XGBoost features built on the canonical sample index — same population, same window.

``fitted._build_features`` derives its lags with
``pl.col("delta_t_min").forward_fill().shift(horizon + k - 1).over(_SLOT_COLS)``.
That shift is **positional**: it steps back `k` rows inside a
``(empresaid, direction, pair_rank)`` slot without checking that consecutive rows
are consecutive minutes. It is the very defect audited in §3 for the DL windows,
reaching the fitted baseline through a different mechanism — so "levelled
competitor" was never quite true: the two families were mis-specified in
different ways over different populations.

This module rebuilds the feature matrix from the shared sample index instead.
Because contract C2 guarantees the ``T_in + horizon`` timestamps of a sample are
consecutive minutes, ``lag_k`` can be read directly off the grid at
``start_ts + (T_in - k)`` — no forward-fill, no positional shift, no silent
bridging of a day boundary.

Consequence, and the point of the exercise: with ``N_LAGS == T_in == 12`` the
XGBoost sees **exactly the twelve observations the LSTM sees**, for exactly the
same set of samples. Levelling stops being an argument and becomes a property.

The atypical flag is absent by design (plan-reentrenamiento.md C3): it is a
whole-day aggregate and therefore not knowable at prediction time.
"""
from __future__ import annotations

import numpy as np
import polars as pl

# Number of lag features. Kept equal to T_in so the fitted baseline and the
# network consume the same window; changing one without the other breaks the
# levelling claim.
N_LAGS: int = 12

FEATURE_NAMES: list[str] = (
    [f"lag_{k}" for k in range(1, N_LAGS + 1)]
    + ["hour", "weekday", "direction", "pair_rank"]
)


def _dense_grid(
    frame: pl.DataFrame, *, max_N: int, value_col: str
) -> tuple[dict, np.ndarray, np.ndarray]:
    """(timestamp -> row) map plus dense value / validity grids for one series."""
    timestamps = frame.select("t").unique().sort("t").get_column("t").to_numpy()
    ts_index = {ts: i for i, ts in enumerate(timestamps)}

    values = np.full((timestamps.size, max_N), np.nan, dtype=np.float64)

    rows = frame.select(["t", "pair_rank", value_col])
    row_of = np.array([ts_index[ts] for ts in rows.get_column("t").to_numpy()])
    pr = rows.get_column("pair_rank").to_numpy()
    val = np.asarray(rows.get_column(value_col).to_numpy(), dtype=np.float64)

    # np.isnan, not polars' is_null: converting a column to numpy turns nulls
    # into NaN and drops the null flag, so is_null answers False for all of them.
    # Harmless here only because the grid defaults to NaN and callers re-check
    # with np.isnan — but relying on that would be luck, not design.
    ok = (pr >= 0) & (pr < max_N) & ~np.isnan(val)
    values[row_of[ok], pr[ok]] = val[ok]

    return ts_index, timestamps, values


def build_contiguous_features(
    frame: pl.DataFrame,
    sample_index: pl.DataFrame,
    *,
    horizon: int,
    T_in: int,
    max_N_by_direction: dict[tuple[int, int], int],
    value_col: str = "delta_t_min",
) -> tuple[np.ndarray, np.ndarray, pl.DataFrame]:
    """Feature matrix, target vector and key frame for one corridor.

    One row per ``(sample, pair_rank)`` whose target and whose ``lag_1``
    (persistence) are both present — the paired set, matching what the DL export
    keeps.

    Returns
    -------
    X : (n_rows, len(FEATURE_NAMES)) float64
    y : (n_rows,) float64 — the target headway in minutes
    keys : DataFrame with empresaid, direction, start_ts, target_ts, horizon,
           pair_rank and ``y_pred_persist`` (== lag_1), in row order of ``X``.
    """
    if N_LAGS > T_in:
        raise ValueError(f"N_LAGS ({N_LAGS}) exceeds T_in ({T_in})")

    blocks_X: list[np.ndarray] = []
    blocks_y: list[np.ndarray] = []
    blocks_key: list[pl.DataFrame] = []

    for (empresaid, direction), idx_part in _partition_index(sample_index):
        max_N = max_N_by_direction[(empresaid, direction)]
        series = frame.filter(
            (pl.col("empresaid") == empresaid) & (pl.col("direction") == direction)
        )
        ts_index, _timestamps, values = _dense_grid(
            series, max_N=max_N, value_col=value_col
        )

        starts = idx_part.get_column("start_ts").to_numpy()
        targets = idx_part.get_column("target_ts").to_numpy()
        start_rows = np.array([ts_index[ts] for ts in starts], dtype=np.int64)
        # C2 makes the run contiguous, so the target sits a fixed offset away.
        target_rows = start_rows + (T_in - 1 + horizon)

        # lag_k is the observation k-1 minutes before the end of the window.
        # k=1 is the last input snapshot, i.e. exactly B1 persistence.
        lag_rows = np.stack(
            [start_rows + (T_in - k) for k in range(1, N_LAGS + 1)], axis=1
        )

        # (n_samples, max_N) -> flattened per pair_rank below.
        y_grid = values[target_rows]                      # (n_samples, max_N)
        lag_grid = values[lag_rows]                       # (n_samples, N_LAGS, max_N)

        keep = ~np.isnan(y_grid) & ~np.isnan(lag_grid[:, 0, :])
        if not keep.any():
            continue
        s_i, pr_i = np.nonzero(keep)

        lags = lag_grid[s_i, :, pr_i]                     # (n_kept, N_LAGS)
        # Remaining lags may still be missing; XGBoost handles NaN natively, so
        # they are passed through rather than imputed (no forward-fill here).

        target_ts = targets[s_i]
        hours = target_ts.astype("datetime64[h]").astype(np.int64) % 24
        # Monday=1 .. Sunday=7, matching polars' dt.weekday().
        days = (
            (target_ts.astype("datetime64[D]").astype(np.int64) + 3) % 7
        ) + 1

        X = np.column_stack(
            [
                lags,
                hours.astype(np.float64),
                days.astype(np.float64),
                np.full(s_i.size, float(direction)),
                pr_i.astype(np.float64),
            ]
        )
        blocks_X.append(X)
        blocks_y.append(y_grid[s_i, pr_i])
        blocks_key.append(
            pl.DataFrame(
                {
                    "empresaid": np.full(s_i.size, empresaid, dtype=np.int64),
                    "direction": np.full(s_i.size, direction, dtype=np.int64),
                    "start_ts": starts[s_i],
                    "target_ts": target_ts,
                    "horizon": np.full(s_i.size, horizon, dtype=np.int64),
                    "pair_rank": pr_i.astype(np.int64),
                    "y_pred_persist": lags[:, 0],
                }
            )
        )

    if not blocks_X:
        return (
            np.zeros((0, len(FEATURE_NAMES))),
            np.zeros(0),
            pl.DataFrame(),
        )

    return (
        np.concatenate(blocks_X),
        np.concatenate(blocks_y),
        pl.concat(blocks_key),
    )


def _partition_index(sample_index: pl.DataFrame):
    """Yield ((empresaid, direction), sub-frame) in deterministic order."""
    parts = sample_index.sort(["empresaid", "direction", "start_ts"]).partition_by(
        ["empresaid", "direction"], maintain_order=True
    )
    for part in parts:
        first = part.row(0, named=True)
        yield (int(first["empresaid"]), int(first["direction"])), part

## Module: evaluation/residual_export

Clave completa + verificación de unicidad.

In [ ]:
"""Canonical per-sample residual export — the full-key contract.

Every question this project could not answer from disk traces to an export that
threw the key away:

  * ``harness.py`` exported XGBoost residuals keyed on ``t`` alone, which is not
    unique (~4.49 rows per ``(t, direction)``). Repairing the DL-vs-XGBoost
    comparison therefore needed a whole new Kaggle kernel, ``20-xgb-paired-export``
    (audit §2.1).
  * The DL residual CSVs carry only
    ``corridor, direction, horizon, y_true, y_pred_dl, y_pred_persist`` — no
    ``t``, no ``pair_rank`` — which is why clustering by service day (#6) and a
    per-position error profile (#5) cannot be computed locally at all.

Both are the same mistake: a lossy export turns every new question into another
GPU run. This module fixes it once, for every model family.

The key
-------
``(corridor, direction, horizon, split, start_ts, target_ts, pair_rank)`` is
unique by construction: the sample index guarantees one row per
``(empresaid, direction, start_ts, horizon)`` (contract C1) and ``pair_rank``
indexes position within that sample's predicted vector.

Reconstruction
--------------
Model outputs arrive as dense ``(n_samples, max_N)`` arrays whose row order is
the sample index's row order and whose column order is ``pair_rank``. The key is
therefore recoverable exactly: repeat each index row ``max_N`` times, tile
``pair_rank`` across it, then drop masked-out cells. No join, no ambiguity.
"""
from __future__ import annotations

import numpy as np
import polars as pl

# Canonical column order. Key first, then values — so a `head` on the CSV shows
# what identifies a row before what it measured.
RESIDUAL_KEY_COLUMNS: list[str] = [
    "corridor",
    "direction",
    "horizon",
    "split",
    "start_ts",
    "target_ts",
    "pair_rank",
]

RESIDUAL_VALUE_COLUMNS: list[str] = [
    "y_true",
    "y_pred_model",
    "y_pred_persist",
]

RESIDUAL_COLUMNS: list[str] = RESIDUAL_KEY_COLUMNS + RESIDUAL_VALUE_COLUMNS


def direction_label(direction_val: int) -> str:
    """Signed direction label ("-1" / "+1").

    Kept identical to ``baselines.harness._direction_label`` and the legacy DL
    exports so the new residuals stay readable by the existing analysis layer.
    """
    return f"+{direction_val}" if direction_val > 0 else str(direction_val)


def build_keyed_residuals(
    sample_index: pl.DataFrame,
    *,
    corridor: str,
    split: str,
    y_true: np.ndarray,
    y_pred_model: np.ndarray,
    y_pred_persist: np.ndarray,
    target_mask: np.ndarray,
    persist_mask: np.ndarray,
) -> pl.DataFrame:
    """Per-sample paired residuals carrying the full key.

    Parameters
    ----------
    sample_index:
        The frame from ``make_sample_index``, in the order the model consumed
        it. Row ``i`` of every array below corresponds to row ``i`` here.
    corridor:
        Corridor label ("E2", "E59", "E4").
    split:
        Split label ("train", "val", "test").
    y_true, y_pred_model, y_pred_persist:
        Dense ``(n_samples, max_N)`` arrays in original (un-z-scored) units.
    target_mask, persist_mask:
        Dense ``(n_samples, max_N)`` boolean arrays, True = VALID (INV-5).
        A cell is exported only where BOTH are valid — the paired set the
        significance tests require.

    Returns
    -------
    DataFrame with ``RESIDUAL_COLUMNS``, sorted deterministically.

    Raises
    ------
    ValueError
        If any array's shape disagrees with the index height, which would mean
        the export is silently misaligned with the population it claims.
    """
    n = sample_index.height
    arrays = {
        "y_true": y_true,
        "y_pred_model": y_pred_model,
        "y_pred_persist": y_pred_persist,
        "target_mask": target_mask,
        "persist_mask": persist_mask,
    }
    shapes = {name: a.shape for name, a in arrays.items()}
    if len({s for s in shapes.values()}) != 1:
        raise ValueError(f"arrays disagree in shape: {shapes}")
    n_rows, max_N = y_true.shape
    if n_rows != n:
        raise ValueError(
            f"array rows ({n_rows}) != sample index height ({n}); the export "
            "is misaligned with its population"
        )

    keep = target_mask & persist_mask
    if not keep.any():
        return pl.DataFrame(schema={c: pl.Utf8 for c in RESIDUAL_COLUMNS})

    # Row i of the flattened arrays maps to index row i // max_N, pair_rank i % max_N.
    rows, cols = np.nonzero(keep)

    directions = sample_index.get_column("direction").to_numpy()[rows]
    starts = sample_index.get_column("start_ts").to_numpy()[rows]
    targets = sample_index.get_column("target_ts").to_numpy()[rows]
    horizons = sample_index.get_column("horizon").to_numpy()[rows]

    frame = pl.DataFrame(
        {
            "corridor": np.full(rows.size, corridor),
            "direction": [direction_label(int(d)) for d in directions],
            "horizon": horizons.astype(np.int64),
            "split": np.full(rows.size, split),
            "start_ts": starts,
            "target_ts": targets,
            "pair_rank": cols.astype(np.int64),
            "y_true": y_true[rows, cols].astype(np.float64),
            "y_pred_model": y_pred_model[rows, cols].astype(np.float64),
            "y_pred_persist": y_pred_persist[rows, cols].astype(np.float64),
        }
    )

    return frame.select(RESIDUAL_COLUMNS).sort(
        ["corridor", "direction", "horizon", "start_ts", "pair_rank"]
    )


def assert_key_is_unique(residuals: pl.DataFrame) -> None:
    """Fail closed when the exported key does not identify a row.

    This is the check whose absence cost a Kaggle kernel: ``harness.py``'s
    docstring declared ``t`` a join key and nothing verified it.
    """
    keys = residuals.select(RESIDUAL_KEY_COLUMNS)
    if keys.height != keys.unique().height:
        dupes = (
            keys.group_by(RESIDUAL_KEY_COLUMNS)
            .len()
            .filter(pl.col("len") > 1)
            .sort("len", descending=True)
        )
        raise ValueError(
            f"residual key is not unique: {dupes.height} duplicated keys, "
            f"worst multiplicity {dupes.get_column('len').max()}"
        )

## Preparación + portón de población compartida

Para cada corredor: split temporal, winsorización con umbral de train, y luego
—por horizonte— se reconstruye el índice canónico y se verifica su dígito. Si
alguno no coincide, la corrida se detiene: significa que esta corrida ya no es
comparable con la del LSTM.

In [ ]:

T_IN = DEFAULT_T_IN  # 12

def _index_digest(index: pl.DataFrame) -> str:
    canonical = index.select(
        ["empresaid", "direction", "start_ts", "target_ts", "horizon"]
    ).sort(["empresaid", "direction", "horizon", "start_ts"])
    payload = canonical.write_csv(datetime_format="%Y-%m-%dT%H:%M:%S")
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

PREPARED, MAXN = {}, {}
for name, emp in CORRIDORS:
    raw = pl.read_parquet(_resolve_input(f"headways_{name}.parquet")).with_columns(
        pl.lit(emp, dtype=pl.Int64).alias("empresaid")
    )
    df_split = split_temporal(raw)
    df_winsor, threshold = winsorize_train_p99(df_split)
    PREPARED[name] = df_winsor
    MAXN[name] = compute_max_N(df_winsor.filter(pl.col("split") == "train"), quantile=0.99)
    print(f"{name}: {raw.height:,} rows, winsor threshold={threshold:.4f} min, max_N={MAXN[name]}")

INDEX = {}
for name, _emp in CORRIDORS:
    for split in ["train", "val", "test"]:
        part = PREPARED[name].filter(pl.col("split") == split)
        for horizon in HORIZONS:
            idx = make_sample_index(part, horizon=horizon, T_in=T_IN)
            got = _index_digest(idx)
            expected = INDEX_DIGESTS[f"{name}|{split}|{horizon}"]
            if got != expected:
                raise ValueError(
                    f"SHARED-POPULATION GATE FAILED for {name}/{split}/h{horizon}: "
                    f"{got} != frozen {expected}. This run is NOT comparable with the "
                    f"LSTM notebooks — stop and re-freeze the manifest."
                )
            INDEX[(name, split, horizon)] = idx
print("\nShared-population gate: PASSED for every corridor x split x horizon.")

## Búsqueda de 24 configuraciones + ajuste final

Selección **estrictamente sobre validación**: el split de test nunca entra en
ninguna `DMatrix` de la búsqueda. La configuración ganadora por
`(corredor, horizonte)` queda registrada para que sea auditable.

In [ ]:

def features_for(name, split, horizon):
    part = PREPARED[name].filter(pl.col("split") == split)
    return build_contiguous_features(
        part, INDEX[(name, split, horizon)],
        horizon=horizon, T_in=T_IN, max_N_by_direction=MAXN[name],
    )

rows, resid_frames, search_rows = [], [], []

for name, _emp in CORRIDORS:
    for horizon in HORIZONS:
        t0 = time.time()
        Xtr, ytr, _ = features_for(name, "train", horizon)
        Xva, yva, _ = features_for(name, "val", horizon)
        Xte, yte, kte = features_for(name, "test", horizon)

        dtr = xgb.DMatrix(Xtr, label=ytr, feature_names=FEATURE_NAMES)
        dva = xgb.DMatrix(Xva, label=yva, feature_names=FEATURE_NAMES)
        dte = xgb.DMatrix(Xte, label=yte, feature_names=FEATURE_NAMES)

        best = None
        for cfg in sample_search_configs(SEARCH_N_CONFIGS, seed=SEARCH_SEED):
            params = {"objective": "reg:squarederror", "seed": 42, "nthread": 4, **cfg}
            booster = xgb.train(
                params, dtr, num_boost_round=800, evals=[(dva, "val")],
                early_stopping_rounds=40, verbose_eval=False,
            )
            score = float(booster.best_score)
            if best is None or score < best[0]:
                best = (score, cfg, booster)

        val_rmse, cfg, booster = best
        pred = booster.predict(dte, iteration_range=(0, booster.best_iteration + 1))
        persist = kte.get_column("y_pred_persist").to_numpy()

        mae_x, rmse_x = mae(yte, pred), rmse(yte, pred)
        mae_p, rmse_p = mae(yte, persist), rmse(yte, persist)
        print(f"{name} h={horizon:2d}: n={len(yte):7,}  MAE_xgb={mae_x:.4f}  "
              f"MAE_pers={mae_p:.4f}  d={mae_x - mae_p:+.4f}  "
              f"val_rmse={val_rmse:.4f}  ({time.time() - t0:.0f}s)", flush=True)

        for model_name, m_mae, m_rmse in [("B5_XGB_CONTIG", mae_x, rmse_x),
                                          ("B1_PERSIST", mae_p, rmse_p)]:
            for metric_name, value in [("MAE", m_mae), ("RMSE", m_rmse)]:
                rows.append({"corridor": name, "horizon": horizon, "baseline": model_name,
                             "metric": metric_name, "value": float(value), "n": len(yte)})

        search_rows.append({"corridor": name, "horizon": horizon, "val_rmse": val_rmse,
                            "best_iteration": booster.best_iteration,
                            "n_configs": SEARCH_N_CONFIGS, "search_seed": SEARCH_SEED,
                            **{f"param_{k}": v for k, v in cfg.items()}})

        n_rows = len(yte)
        resid_frames.append(pl.DataFrame({
            "corridor": [name] * n_rows,
            "direction": [f"+{d}" if d > 0 else str(d)
                          for d in kte.get_column("direction").to_list()],
            "horizon": [horizon] * n_rows,
            "split": ["test"] * n_rows,
            "start_ts": kte.get_column("start_ts"),
            "target_ts": kte.get_column("target_ts"),
            "pair_rank": kte.get_column("pair_rank"),
            "y_true": yte.astype("float64"),
            "y_pred_model": pred.astype("float64"),
            "y_pred_persist": persist.astype("float64"),
        }).select(RESIDUAL_COLUMNS))

results = pl.DataFrame(rows)
results.write_csv(RESULTS_OUT)
print(f"\nResults: {RESULTS_OUT} ({results.height} rows)")
print(results)

search = pl.DataFrame(search_rows)
search.write_csv(SEARCH_OUT)
print(f"Search provenance: {SEARCH_OUT} ({search.height} rows)")

residuals = pl.concat(resid_frames)
assert_key_is_unique(residuals)
residuals.write_csv(RESID_OUT)
print(f"Residuals: {RESID_OUT} ({residuals.height:,} rows, key verified unique)")